# Capítulo 9 — Validação e medição

**Livro:** começa na página **325** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py: modelos via OpenRouter, com controle de gasto
from ia import *
import json, re, time, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO)

## Component Evaluation · PDF p. 331

#### Exercício: Avaliar a escolha de ferramenta

Monte 6 casos de teste (pedido → ferramenta esperada) e meça a acurácia do modelo na **escolha** da ferramenta (primeira `tool_call`), sem executar nada.

In [ ]:
CATALOGO = {"teclado": 199.9, "mouse": 89.9, "monitor": 1299.0, "cadeira": 899.0}
PEDIDOS = {"A100": {"status": "enviado", "previsao": "2025-06-12"}, "A101": {"status": "em separação", "previsao": "2025-06-15"}}
def consultar_preco(produto: str) -> str:
    """Devolve o preço de um produto do catálogo."""
    p = CATALOGO.get(produto.lower())
    return f"{produto}: R$ {p:.2f}" if p else f"{produto} não está no catálogo"
def status_pedido(pedido_id: str) -> str:
    """Devolve o status e a previsão de entrega de um pedido."""
    p = PEDIDOS.get(pedido_id.upper())
    return json.dumps(p, ensure_ascii=False) if p else "pedido não encontrado"
FUNCOES = {"consultar_preco": consultar_preco, "status_pedido": status_pedido}
TOOLS = [{"type": "function", "function": {"name": "consultar_preco", "description": "Preço de um produto do catálogo",
          "parameters": {"type": "object", "properties": {"produto": {"type": "string"}}, "required": ["produto"]}}},
         {"type": "function", "function": {"name": "status_pedido", "description": "Status e previsão de entrega de um pedido",
          "parameters": {"type": "object", "properties": {"pedido_id": {"type": "string"}}, "required": ["pedido_id"]}}}]
casos = [("quanto custa o mouse?", "consultar_preco"), ("cadê meu pedido A101?", "status_pedido"),
         ("qual o valor do monitor?", "consultar_preco"), ("o A100 já saiu para entrega?", "status_pedido"),
         ("preço da cadeira", "consultar_preco"), ("previsão do pedido A101", "status_pedido")]
acertos = 0
for q, esperado in casos:
    r = conversar([{"role": "user", "content": q}], tools=TOOLS, tool_choice="required", temperature=0).choices[0].message
    escolhido = r.tool_calls[0].function.name; acertos += escolhido == esperado
    print(q, "->", escolhido)
print("acurácia:", acertos / len(casos))

## Holistic Evaluation · PDF p. 338

#### Exercício: Avaliar a resposta final com um juiz

Peça ao modelo forte para avaliar, de 1 a 5, se a resposta final de um agente **resolveu o pedido** e se **inventou algo** que não estava nas observações das ferramentas. Devolva JSON.

In [ ]:
obs = "consultar_preco -> teclado: R$ 199.90 | status_pedido -> {'status': 'enviado', 'previsao': '2025-06-12'}"
resposta = "O teclado custa R$ 199,90 e seu pedido A100 foi enviado, chega dia 12/06 com frete grátis."
juiz = perguntar(f'Observações das ferramentas: {obs}\nResposta do agente: {resposta}\nAvalie em JSON: {{"resolveu": 1-5, "inventou": true/false, "motivo": "..."}}',
                 modelo=MODELO_FORTE, response_format={"type": "json_object"}, temperature=0)
print(json.loads(juiz))  # o "frete grátis" não estava nas observações